# **Atividade Prática**
<font size=3>

- **Tema:** fluxo de trabalho.
- **Prazo de entrega:** 29 de Abril.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Questão:**
<font size=3>

Com base no *dataset* de [review de skincare](https://www.kaggle.com/datasets/nenamalikah/nlp-ulta-skincare-reviews), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe o *dataset* e verifique se existe algum valor `NaN`, usando o método `df.isnull().any()`. Caso exita, verifique a quantidade de linhas com esse problema, com `df.isnull().any().sum()`. Se forem poucas linhas de dados, remova as linhas com `df = df.dropna()`.
2. Defina o atributo `Verified_Buyer` como a variável alvo (`y`) e o atributo de *reviews* como `texts`;
3. Examine alguns poemas de `texts` a fim de definir as etapas de pré-processamento;
4. Divida o *dataframe* entre dados de treinamento e teste;
5. Transforme as categorias das variáveis `y_train` e `y_test` em representações numéricas com a classe [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html);
6. Com base no **_notebook_ 8**, defina o objeto do [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) a fim de encadear todo o pré-processamento do [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) — com os dados de **textos**, **categóricos** e **numéricos** — junto ao modelo [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html);
7. Encontre o melhor pré-processamento e modelo, utilizando o [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html), para o hiperparâmetros `n_components` (do [`TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html)) e `n_estimators` (número de árvores);
8. Realize a **avaliação** com a função [`classification_report`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html). **Escreva** em um **célula markdown** sua interpretação dos resultados.


## Resolução

### Investigação preliminar

#### Imports e parametros gerais

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus   import stopwords
from nltk.corpus   import wordnet
from nltk.tokenize import word_tokenize
from nltk.stem     import WordNetLemmatizer
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, normalize

In [2]:
MIN_DF=5 # mínimo de documentos que uma palavra deve aparecer para ser considerada
MAX_DF=0.95 # máximo de documentos que uma palavra deve aparecer para ser considerada
RANDOM_STATE=42 # semente para garantir a reprodutibilidade dos resultados
TEST_SIZE=0.10 # proporção de dados para teste
SVD_COMPONENTS=25 # número de componentes para o SVD

## Configurações gerais do pandas

pd.set_option("display.max_colwidth", 500)   # não truncar texto
pd.set_option("display.max_columns", None)    # mostrar todas as colunas
pd.set_option("display.width", 120)             # deixa o pandas ajustar a largura


In [3]:
# Faz o download da lista de stop words:
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/gilcesarf/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


#### Leitura do dataset

In [4]:
df = pd.read_csv("./dataset/skincare_reviews.csv") # estou executando no meu notebook pessoal, não no google colab.

print(df.shape)

df.head()

(4150, 10)


,Review_Title,Review_Text,Verified_Buyer,Review_Date,Review_Location,Review_Upvotes,Review_Downvotes,Product,Brand,Scrape_Date
0,Perfect,Love using this on my face while in the shower. Heats up and gives a light scrub nicely,No,15 days ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
1,You need this,Even better than the daily microfoliant. I'm obsessed. My skin is SO MUCH smoother,No,27 days ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
2,Clean skin,Enjoy this product so much ! I look forward to using it - really feels great.,No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
3,Love This Stuff!,I've never tried anything like this before and I love it. When you apply it to your face you get a little shot of warm that feels so good. The scrub seems very gritty but the only side effects I've encountered have been positive ones.,No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23
4,This exfoliates very nicely and,"This exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin . I highly recommended it, i will buy it again.",No,2 months ago,Undisclosed,0,0,Multi-Vitamin Thermafoliant,Dermalogica,3/27/23


#### Investigando nulos

In [5]:
df.isnull().any()

Review_Title        False
Review_Text          True
Verified_Buyer      False
Review_Date         False
Review_Location      True
Review_Upvotes      False
Review_Downvotes    False
Product             False
Brand               False
Scrape_Date         False
dtype: bool

In [6]:
print(f"Quantidade de linhas com valores null: {df.isnull().any(axis=1).sum().item()}")

Quantidade de linhas com valores null: 4


##### Quais são as linhas com valor nulo:

In [7]:
df[df.isnull().any(axis=1)]

,Review_Title,Review_Text,Verified_Buyer,Review_Date,Review_Location,Review_Upvotes,Review_Downvotes,Product,Brand,Scrape_Date
3397,Half full product,"The only reason I'm rating this three stars is because it's already a travel-size item, do they really needs to only fill the bottle up halfway? For how pricey this travel size bottle is, they could at least fill the whole thing up or use a smaller container, because otherwise it seems misleading.",No,2 years ago,NaN,2,0,Daily Microfoliant,Dermalogica,3/27/23
3560,I would definitely buy this product again,NaN,Yes,3 years ago,MN,0,1,Daily Microfoliant,Dermalogica,3/27/23
3684,Received a sample and loved it!,NaN,Yes,4 years ago,"Columbia, SC",0,0,Daily Microfoliant,Dermalogica,3/27/23
3686,This product works,NaN,Yes,4 years ago,"Columbia, SC",0,0,Daily Microfoliant,Dermalogica,3/27/23


##### Limpando os nulos (no mesmo dataset)

In [8]:
df.dropna(inplace=True)

##### Conferindo a limpeza

In [9]:
print(f"Quantidade de linhas com valores null: {df.isnull().any(axis=1).sum().item()}")

Quantidade de linhas com valores null: 0


#### Coletando algumas amostras de textos

In [10]:
cols_str = df.select_dtypes(include=["object", "string"]).columns
n = 10
amostra = (
    pd.concat([
        df.head(5),
        df.tail(5),
        df.sample(n=n, random_state=RANDOM_STATE),
    ])
    .drop_duplicates()  # evita repetir linhas se cair na amostra/heads/tails
)
for i, row in amostra.iterrows():
    print(f"Review (Title + Text): {row["Review_Title"]} {row["Review_Text"]}")


Review (Title + Text): Perfect Love using this on my face while in the shower. Heats up and gives a light scrub nicely
Review (Title + Text): You need this Even better than the daily microfoliant. I'm obsessed. My skin is SO MUCH smoother
Review (Title + Text): Clean skin Enjoy this product so much ! I look forward to using it - really feels great.
Review (Title + Text): Love This Stuff! I've never tried anything like this before and I love it. When you apply it to your face you get a little shot of warm that feels so good. The scrub seems very gritty but the only side effects I've encountered have been positive ones.
Review (Title + Text): This exfoliates very nicely and This exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin . I highly recommended it, i will buy it again.
Review (Title + Text): I would buy this product again Much better product than a scrub.
Review (Title + Text): Gentle exfoliant- leaves skin smooth & soft I've b

#### Definindo uma função de pré-processamento

##### Função de expansão de contrações

In [11]:
def expand_contractions_tokens(tokens):
    """
    Expande contrações tokenizadas pelo NLTK e remove possessivo 's.

    Exemplos:
    ["i", "'m"]       -> ["i", "am"]
    ["ca", "n't"]     -> ["can", "not"]
    ["wo", "n't"]     -> ["will", "not"]
    ["you", "'re"]    -> ["you", "are"]
    ["skin", "'s"]    -> ["skin"]
    ["product", "'s"] -> ["product"]
    """

    suffix_map = {
        "'m": ["am"],
        "'re": ["are"],
        "'ve": ["have"],
        "'ll": ["will"],
        "'d": ["would"],  # ambíguo: would/had
    }

    negation_base = {
        "ca": "can",
        "wo": "will",
        "do": "do",
        "does": "does",
        "did": "did",
        "is": "is",
        "are": "are",
        "was": "was",
        "were": "were",
        "has": "has",
        "have": "have",
        "had": "had",
        "could": "could",
        "should": "should",
        "would": "would",
        "must": "must",
        "need": "need",
    }

    expanded = []
    i = 0

    while i < len(tokens):
        token = tokens[i]

        if i + 1 < len(tokens):
            next_token = tokens[i + 1]

            # can't / won't / didn't / isn't ...
            if next_token == "n't" and token in negation_base:
                expanded.extend([negation_base[token], "not"])
                i += 2
                continue

            # i'm / you're / they've / i'll / i'd ...
            if next_token in suffix_map:
                expanded.append(token)
                expanded.extend(suffix_map[next_token])
                i += 2
                continue

            # possessivo ou 's ambíguo:
            # skin 's -> skin
            # product 's -> product
            # it 's -> it
            #
            # Decisão intencional
            # não expandir 's para "is"/"has" para evitar erro semantico.
            if next_token == "'s":
                expanded.append(token)
                i += 2
                continue

        # remove tokens soltos que são apenas fragmentos de contração
        if token in {"n't", "'m", "'re", "'ve", "'ll", "'d", "'s"}:
            i += 1
            continue

        expanded.append(token)
        i += 1

    return expanded


##### Função de mapeamento de classes gramaticais

In [12]:
    
def penn_to_wordnet_pos(tag: str):
    """
    Converte uma tag POS (classe gramatical) do Penn Treebank (saída do nltk.pos_tag) 
    para uma categoria do WordNet compatível com o WordNetLemmatizer.
    Args:
        tag (str): Tag POS do Penn Treebank, por exemplo "NN", "VB", "JJ", "RB".
    Returns:
        str | None: Uma das categorias do WordNet (wordnet.NOUN, wordnet.VERB, wordnet.ADJ, wordnet.ADV)
        ou None quando a tag não pertence a {J, V, N, R}.
    Notes:
        - Mapeamento:
          - J* -> ADJ
          - V* -> VERB
          - N* -> NOUN
          - R* -> ADV
        - Retornar None é uma abordagem conservadora: se o POS não for reconhecido, não força
          a lematização com uma classe incorreta.
    """
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("N"):
        return wordnet.NOUN
    if tag.startswith("R"):
        return wordnet.ADV
    return None

##### Função de pre-processamento

In [13]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, lemma=False, remove_stopwords=False):
    """
    Normaliza um texto com limpeza básica, remoção de stopwords e lematização opcional com POS-tagging.
    Etapas:
        - Converte para minúsculas.
        - Remove pontuação/caracteres não alfanuméricos, dígitos e espaços extras.
        - Tokeniza por espaços.
        - Remove stopwords (variável externa stop_words).
        - Se lemma=True, aplica POS-tagging (nltk.pos_tag), mapeia POS para WordNet e lematiza
          com WordNetLemmatizer de forma conservadora (só lematiza quando há POS mapeável).
    Args:
        text (str): Texto de entrada.
        lemma (bool): Se True, aplica lematização com POS. Se False, não lematiza.
    Returns:
        str: Texto processado, com tokens separados por um único espaço.
    Dependencies:
        - re, nltk
        - stop_words (iterável/set definido externamente)
        - lemmatizer (WordNetLemmatizer definido externamente)
        - penn_to_wordnet_pos(tag)
    """
    text = text.lower()                               # converte para minúsculas

    # mantém apóstrofo para contrações (don't, i'm, you're) e remove o resto
    text = re.sub(r"[^a-z0-9'\s]", " ", text)  # substitui caracteres não alfanuméricos por espaços
    text = re.sub(r"\s+", " ", text)           # substitui múltiplos espaços por um único espaço
    text = re.sub(r"\d+", "", text)            # remove dígitos
    text = text.strip()                        # remove espaços em branco no início e no final

    # vamos testar com word_tokenize do nltk ao inves de regex propria para testar.
    # por debaixo da função utiliza o codigo da seguinte fonte de referencia: 
    # https://www.nltk.org/_modules/nltk/tokenize/destructive.html#NLTKWordTokenizer
    tokens = word_tokenize(text)               # tokeniza usando nltk pois ele lida melhor com contrações

    tokens = [t for t in tokens
              if re.search(r"[a-z]", t)]       # remove tokens que não possuem letras

    if remove_stopwords:
        tokens = [token for token in tokens
                  if token not in stop_words]    # remove stop words

    tokens = expand_contractions_tokens(tokens)

    # remove apóstrofos residuais depois da expansão
    tokens = [re.sub(r"'", "", token) for token in tokens]

    if lemma:
        tagged = nltk.pos_tag(tokens)
        tokens = [# aplica lematização se houver POS (classe gramatical) mapeável
                  (lemmatizer.lemmatize(token, pos=wn_pos) if wn_pos else token)  
                  for token, tag in tagged
                  # truque para invocar penn_to_wordnet_pos e nomear resultado como wn_pos
                  for wn_pos in [penn_to_wordnet_pos(tag)]]

    return " ".join(tokens)

##### Verificando contrações restantes para melhorar função de expansão

In [ ]:
def tem_apostrofo(token: str) -> bool:
    return ("'" in token) or ("’" in token)


def triplas_contracao_de_tokens_processados(processed: str):
    # processed: saída de preprocess_text (tokens separados por espaço)
    tokens = processed.split()
    vistos_locais = set()
    for i, tok in enumerate(tokens):
        if not tem_apostrofo(tok):
            continue
        prev_tok = tokens[i - 1] if i > 0 else None
        next_tok = tokens[i + 1] if i + 1 < len(tokens) else None
        vistos_locais.add((prev_tok, tok, next_tok))
    return vistos_locais


unicos = set()

for _, row in df.iterrows():
    texto = f"{row['Review_Title']} {row['Review_Text']}"
    proc = preprocess_text(texto, lemma=False, remove_stopwords=False)  # ajuste conforme seu preprocessor
    unicos |= triplas_contracao_de_tokens_processados(proc)

# saída única (ordenada para ficar estável)
saida_unica = sorted(unicos, key=lambda t: (t[0] or "", t[1], t[2] or ""))

print(f"triplas encontradas: {len(saida_unica)}")
for tripla in saida_unica:
    print(tripla)

triplas encontradas: 0


#### Revisando textos resultantes

##### Sem lematização e sem remoção de stop words

In [15]:

for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}"))


Review (Title + Text): perfect love using this on my face while in the shower heats up and gives a light scrub nicely
Review (Title + Text): you need this even better than the daily microfoliant i am obsessed my skin is so much smoother
Review (Title + Text): clean skin enjoy this product so much i look forward to using it really feels great
Review (Title + Text): love this stuff i have never tried anything like this before and i love it when you apply it to your face you get a little shot of warm that feels so good the scrub seems very gritty but the only side effects i have encountered have been positive ones
Review (Title + Text): this exfoliates very nicely and this exfoliates very nicely and gives a very smooth skin after with no irritation and no reaction to the skin i highly recommended it i will buy it again
Review (Title + Text): i would buy this product again much better product than a scrub
Review (Title + Text): gentle exfoliant leaves skin smooth soft i have been using thi

##### Com lematização e com remoção de stop words

In [16]:
for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}", lemma=True, remove_stopwords=True))


Review (Title + Text): perfect love use face shower heats give light scrub nicely
Review (Title + Text): need even well daily microfoliant be obsess skin much smoother
Review (Title + Text): clean skin enjoy product much look forward use really feel great
Review (Title + Text): love stuff have never try anything like love apply face get little shot warm feel good scrub seem gritty side effect have encounter positive one
Review (Title + Text): exfoliates nicely exfoliate nicely give smooth skin irritation reaction skin highly recommended buy
Review (Title + Text): would buy product much good product scrub
Review (Title + Text): gentle exfoliant leave skin smooth soft have use exfoliant month depend condition skin will use anywhere week be clear daily be break stuff waay gentle exfoliants honestly find make skin bad harsh abrasive microfoliant none exfoliate smoothly gently often need without harsh feeling like strip skin use skin feel smooth soft calm never irritate tight dry be huge fa

##### Sem lematização e com remoção de stop words

In [17]:
for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}", lemma=False, remove_stopwords=True))

Review (Title + Text): perfect love using face shower heats gives light scrub nicely
Review (Title + Text): need even better daily microfoliant am obsessed skin much smoother
Review (Title + Text): clean skin enjoy product much look forward using really feels great
Review (Title + Text): love stuff have never tried anything like love apply face get little shot warm feels good scrub seems gritty side effects have encountered positive ones
Review (Title + Text): exfoliates nicely exfoliates nicely gives smooth skin irritation reaction skin highly recommended buy
Review (Title + Text): would buy product much better product scrub
Review (Title + Text): gentle exfoliant leaves skin smooth soft have using exfoliant months depending condition skin will use anywhere week am clear daily am breaking stuff waay gentle exfoliants honestly found making skin worse harsh abrasives microfoliant none exfoliates smoothly gently often need without harsh feeling like stripping skin use skin feels smooth s

##### Com lematização e sem remoção de stop words

In [18]:
for i, row in amostra.iterrows():
    print("Review (Title + Text):", preprocess_text(f"{row['Review_Title']} {row['Review_Text']}", lemma=True, remove_stopwords=False))


Review (Title + Text): perfect love use this on my face while in the shower heat up and give a light scrub nicely
Review (Title + Text): you need this even good than the daily microfoliant i be obsess my skin be so much smoother
Review (Title + Text): clean skin enjoy this product so much i look forward to use it really feel great
Review (Title + Text): love this stuff i have never try anything like this before and i love it when you apply it to your face you get a little shot of warm that feel so good the scrub seem very gritty but the only side effect i have encounter have be positive one
Review (Title + Text): this exfoliate very nicely and this exfoliate very nicely and give a very smooth skin after with no irritation and no reaction to the skin i highly recommend it i will buy it again
Review (Title + Text): i would buy this product again much good product than a scrub
Review (Title + Text): gentle exfoliant leave skin smooth soft i have be use this exfoliant for a few month now d

#### Verificando contrações restantes para melhorar função de expansão

In [19]:
def _tem_apostrofo(token: str) -> bool:
    return ("'" in token) or ("’" in token)


def triplas_contracao_de_tokens_processados(processed: str):
    # processed: saída de preprocess_text (tokens separados por espaço)
    tokens = processed.split()
    vistos_locais = set()
    for i, tok in enumerate(tokens):
        if not _tem_apostrofo(tok):
            continue
        prev_tok = tokens[i - 1] if i > 0 else None
        next_tok = tokens[i + 1] if i + 1 < len(tokens) else None
        vistos_locais.add((prev_tok, tok, next_tok))
    return vistos_locais


unicos = set()

for _, row in df.iterrows():
    texto = f"{row['Review_Title']} {row['Review_Text']}"
    proc = preprocess_text(texto, lemma=False, remove_stopwords=False)  # ajuste conforme seu preprocessor
    unicos |= triplas_contracao_de_tokens_processados(proc)

# saída única (ordenada para ficar estável)
saida_unica = sorted(unicos, key=lambda t: (t[0] or "", t[1], t[2] or ""))

print(f"triplas encontradas: {len(saida_unica)}")
for tripla in saida_unica:
    print(tripla)

triplas encontradas: 0
